# GRPO Smoke Run Analysis

Use this notebook on Great Lakes after a GRPO smoke run finishes. It inspects the output directory, parser behavior, saved completions, and Slurm logs.

In [ ]:
from pathlib import Path
import json
import re

RUN_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/ckpts/smoke_qwen2_5_0_5b_grpo_100step')
LOG_DIR = Path('/scratch/huterer_root/huterer0/jiamingp/pqs/repos/posttrain-quant-serve/logs')
RUN_DIR, RUN_DIR.exists(), LOG_DIR.exists()

## Check Saved Artifacts

In [ ]:
if RUN_DIR.exists():
    for path in sorted(RUN_DIR.iterdir()):
        size = path.stat().st_size if path.is_file() else None
        print(path.name, 'dir' if path.is_dir() else f'{size/1024**2:.1f} MiB')
else:
    print('Missing RUN_DIR:', RUN_DIR)

In [ ]:
if RUN_DIR.exists():
    print('Checkpoints:')
    for path in sorted(RUN_DIR.glob('checkpoint-*')):
        print(' ', path)

## Inspect Completion Logs

TRL may write completions in different formats across versions. This cell lists what exists under `completions/`.

In [ ]:
completion_dir = RUN_DIR / 'completions'
if completion_dir.exists():
    for path in sorted(completion_dir.rglob('*')):
        if path.is_file():
            print(path, f'{path.stat().st_size/1024:.1f} KiB')
else:
    print('No completions dir found yet:', completion_dir)

## Test Parser On Observed Text

Paste any suspicious model completion into `sample_completion` and check what answer the reward parser extracts.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists() and (REPO_ROOT.parent / 'scripts').exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / 'scripts'))
from gsm8k_reward import extract_model_answer, gsm8k_exact_match_reward

sample_completion = '''Final answer: 312 pages per year.'''
sample_reference = '... #### 624'

print('parsed model answer:', extract_model_answer(sample_completion))
print('reward:', gsm8k_exact_match_reward([sample_completion], [sample_reference]))

## Parse Slurm Logs

Set `JOB_ID` to the 100-step job id. This pulls rough training summary lines and reward table snippets from the logs.

In [ ]:
JOB_ID = ''  # e.g. '509xxxxx'

if JOB_ID:
    out_files = sorted(LOG_DIR.glob(f'*-{JOB_ID}.out'))
    err_files = sorted(LOG_DIR.glob(f'*-{JOB_ID}.err'))
else:
    out_files = sorted(LOG_DIR.glob('pqs-grpo-100-*.out'))[-3:]
    err_files = sorted(LOG_DIR.glob('pqs-grpo-100-*.err'))[-3:]

print('OUT files:', out_files)
print('ERR files:', err_files)

In [ ]:
for path in out_files:
    print('\n===', path.name, '===')
    text = path.read_text(errors='replace')
    for line in text.splitlines():
        if 'train_runtime' in line or 'Step ' in line or 'gsm8k_exact_match' in line:
            print(line[:240])

In [ ]:
for path in err_files:
    print('\n===', path.name, 'tail ===')
    lines = path.read_text(errors='replace').splitlines()
    for line in lines[-80:]:
        print(line)